# SageMaker Endpoint Backtest

Simulates the ML momentum trading strategy by calling SageMaker real-time endpoints.
Iterates through production data day-by-day, calling 6 deployed models (3 architectures x 2 tickers)
to generate trading signals.

**Signal composition:** 70% model majority vote + 15% EMA + 15% McClellan oscillator

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import talib
import boto3
import matplotlib.pyplot as plt
from pathlib import Path

REGION = "us-east-1"
BUCKET = "labs-usd-01"
PREFIX = "AAI_540_group_8"
ROLE_ARN = "arn:aws:iam::806081623304:role/LabRole"
INFERENCE_IMAGE = (
    "763104351884.dkr.ecr.us-east-1.amazonaws.com/"
    "pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
)

TICKERS = ["CL=F", "GC=F"]
MODEL_NAMES = ["lstm", "transformer", "bilstm_attention"]

FEATURE_COLUMNS = [
    "high", "low", "open", "volume",
    "MA", "EMA", "KAMA", "WMA", "MidPrice",
    "BOP", "CMO", "MFI", "ROC", "WILLR",
    "AD", "OBV", "NATR", "ATR", "TRANGE", "TSF",
]

LOOKBACK = 20
WARMUP_DAYS = 60
CAPITAL = 10_000.0
POSITION_SIZE = 100
EMA_SPAN = 10

sm_client = boto3.client("sagemaker", region_name=REGION)
runtime_client = boto3.client("sagemaker-runtime", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)

## Step 1: Deploy Endpoints

In [ ]:
def get_model_package_arn(group_name, model_name):
    """Get the latest approved model package ARN from a group matching model_name."""
    resp = sm_client.list_model_packages(
        ModelPackageGroupName=group_name,
        ModelApprovalStatus="Approved",
        SortBy="CreationTime",
        SortOrder="Descending",
    )
    for pkg in resp["ModelPackageSummaryList"]:
        pkg_detail = sm_client.describe_model_package(ModelPackageName=pkg["ModelPackageArn"])
        meta = pkg_detail.get("CustomerMetadataProperties", {})
        if meta.get("model_architecture") == model_name:
            return pkg["ModelPackageArn"]
    raise ValueError(f"No package found for {model_name} in {group_name}")


def create_endpoint_from_package(endpoint_name, model_package_arn):
    """Create a SageMaker model + endpoint config + endpoint from a model package."""
    model_name = f"{endpoint_name}-model"
    config_name = f"{endpoint_name}-config"

    # Create model from package
    try:
        sm_client.create_model(
            ModelName=model_name,
            ExecutionRoleArn=ROLE_ARN,
            Containers=[{"ModelPackageName": model_package_arn}],
        )
        print(f"  Created model: {model_name}")
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" in str(e):
            print(f"  Model already exists: {model_name}")
        else:
            raise

    # Create endpoint config
    try:
        sm_client.create_endpoint_config(
            EndpointConfigName=config_name,
            ProductionVariants=[{
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": "ml.m5.large",
                "InitialInstanceCount": 1,
            }],
        )
        print(f"  Created config: {config_name}")
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" in str(e):
            print(f"  Config already exists: {config_name}")
        else:
            raise

    # Create endpoint
    try:
        sm_client.create_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name,
        )
        print(f"  Creating endpoint: {endpoint_name}")
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" in str(e):
            print(f"  Endpoint already exists: {endpoint_name}")
        else:
            raise


def wait_for_endpoints(endpoint_names, poll_interval=30):
    """Wait for all endpoints to reach InService status."""
    print(f"\nWaiting for {len(endpoint_names)} endpoints to be InService...")
    pending = set(endpoint_names)

    while pending:
        for name in list(pending):
            resp = sm_client.describe_endpoint(EndpointName=name)
            status = resp["EndpointStatus"]
            if status == "InService":
                print(f"  {name}: InService")
                pending.remove(name)
            elif status == "Failed":
                reason = resp.get("FailureReason", "Unknown")
                raise RuntimeError(f"Endpoint {name} failed: {reason}")
            else:
                print(f"  {name}: {status}...")

        if pending:
            time.sleep(poll_interval)

    print("All endpoints are InService!")

In [ ]:
endpoint_names = []
groups = {"CL=F": "futures-cl-models", "GC=F": "futures-gc-models"}

for ticker in TICKERS:
    group_name = groups[ticker]
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()

    for model_name in MODEL_NAMES:
        endpoint_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"
        print(f"\nDeploying {endpoint_name}...")

        pkg_arn = get_model_package_arn(group_name, model_name)
        print(f"  Package ARN: {pkg_arn}")

        create_endpoint_from_package(endpoint_name, pkg_arn)
        endpoint_names.append(endpoint_name)

wait_for_endpoints(endpoint_names)

## Step 2: Load Production Data

In [ ]:
prod_data = {}
for ticker in TICKERS:
    s3_key = f"{PREFIX}/dataset/prod/{ticker}x_prod.csv"
    local_path = f"/tmp/{ticker}x_prod.csv"
    s3_client.download_file(BUCKET, s3_key, local_path)

    df = pd.read_csv(local_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").sort_index()
    prod_data[ticker] = df
    print(f"{ticker}: {len(df)} rows, {df.index[0].date()} to {df.index[-1].date()}")

## Step 3: Feature Engineering

In [ ]:
def compute_talib_features(ohlcv_df):
    """Compute TA-Lib features from OHLCV data."""
    df = ohlcv_df.copy().sort_index()

    df["MA"] = talib.MA(df["close"], timeperiod=10)
    df["EMA"] = talib.EMA(df["close"], timeperiod=10)
    df["KAMA"] = talib.KAMA(df["close"], timeperiod=10)
    df["WMA"] = talib.WMA(df["close"], timeperiod=10)
    df["MidPrice"] = talib.MIDPRICE(df["high"], df["low"], timeperiod=10)

    df["BOP"] = talib.BOP(df["open"], df["high"], df["low"], df["close"])
    df["CMO"] = talib.CMO(df["close"], timeperiod=10)
    df["MFI"] = talib.MFI(df["high"], df["low"], df["close"], df["volume"])
    df["ROC"] = talib.ROC(df["close"], timeperiod=10)
    df["WILLR"] = talib.WILLR(df["high"], df["low"], df["close"], timeperiod=14)

    df["AD"] = talib.AD(df["high"], df["low"], df["close"], df["volume"])
    df["OBV"] = talib.OBV(df["close"], df["volume"])

    df["NATR"] = talib.NATR(df["high"], df["low"], df["close"], timeperiod=14)
    df["ATR"] = talib.ATR(df["high"], df["low"], df["close"], timeperiod=14)
    df["TRANGE"] = talib.TRANGE(df["high"], df["low"], df["close"])

    df["TSF"] = talib.TSF(df["close"], timeperiod=14)

    df = df.iloc[15:]
    return df


def ema_update(prev, value, span):
    """Incremental EMA update."""
    alpha = 2.0 / (span + 1)
    return value if prev is None else alpha * value + (1 - alpha) * prev

## Step 4: Run Backtest Simulation

In [ ]:
# Build endpoint name mapping
endpoint_map = {}
for ticker in TICKERS:
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()
    endpoint_map[ticker] = {}
    for model_name in MODEL_NAMES:
        ep_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"
        endpoint_map[ticker][model_name] = ep_name

# Initialize tracking
cash = CAPITAL
positions = {t: 0 for t in TICKERS}
portfolio_history = []
trade_log = []

# EMA and McClellan state
ema_prev = {t: None for t in TICKERS}
mcclel_fast = {t: None for t in TICKERS}
mcclel_slow = {t: None for t in TICKERS}
prev_close = {t: None for t in TICKERS}

# Get all trading dates (intersection of both tickers)
all_dates = prod_data[TICKERS[0]].index
for t in TICKERS[1:]:
    all_dates = all_dates.intersection(prod_data[t].index)
all_dates = all_dates.sort_values()

print(f"Total trading days: {len(all_dates)}")
print(f"Warmup period: {WARMUP_DAYS} days")
print(f"Active trading days: {len(all_dates) - WARMUP_DAYS}")

for day_idx, date in enumerate(all_dates):
    # Skip warmup period
    if day_idx < WARMUP_DAYS:
        # Track portfolio value during warmup
        port_val = cash
        for t in TICKERS:
            if positions[t] > 0:
                port_val += positions[t] * prod_data[t].loc[date, "close"]
        portfolio_history.append({"date": date, "portfolio_value": port_val, "cash": cash})
        continue

    for ticker in TICKERS:
        # Get 60-day OHLCV window ending at current date
        ticker_data = prod_data[ticker]
        date_loc = ticker_data.index.get_loc(date)
        if date_loc < 59:
            continue

        window = ticker_data.iloc[date_loc - 59:date_loc + 1].copy()
        ohlcv = pd.DataFrame({
            "open": window["open"].values,
            "high": window["high"].values,
            "low": window["low"].values,
            "close": window["close"].values,
            "volume": window["volume"].values.astype(float),
        })

        # Compute features
        featured = compute_talib_features(ohlcv)
        if len(featured) < LOOKBACK:
            continue

        features = featured[FEATURE_COLUMNS].values[-LOOKBACK:]
        today_close = float(window["close"].iloc[-1])
        yesterday_close = float(window["close"].iloc[-2])

        # --- Model majority vote (70% weight) ---
        votes = 0
        predictions = {}
        for model_name in MODEL_NAMES:
            ep_name = endpoint_map[ticker][model_name]
            payload = json.dumps({"features": features.tolist()})
            resp = runtime_client.invoke_endpoint(
                EndpointName=ep_name,
                ContentType="application/json",
                Body=payload,
            )
            result = json.loads(resp["Body"].read().decode())
            pred_price = result["predicted_price"]
            predictions[model_name] = pred_price
            if pred_price > yesterday_close:
                votes += 1

        model_signal = 1.0 if votes >= 2 else -1.0

        # --- EMA signal (15% weight) ---
        ema_prev[ticker] = ema_update(ema_prev[ticker], today_close, EMA_SPAN)
        ema_signal = 1.0 if today_close > ema_prev[ticker] else -1.0

        # --- McClellan oscillator (15% weight) ---
        daily_change = 0.0
        if prev_close[ticker] is not None:
            daily_change = today_close - prev_close[ticker]
        prev_close[ticker] = today_close

        mcclel_fast[ticker] = ema_update(mcclel_fast[ticker], daily_change, 19)
        mcclel_slow[ticker] = ema_update(mcclel_slow[ticker], daily_change, 39)
        mcclellan = mcclel_fast[ticker] - mcclel_slow[ticker]
        mcclellan_signal = 1.0 if mcclellan > 0 else -1.0

        # --- Combined signal ---
        combined = 0.70 * model_signal + 0.15 * ema_signal + 0.15 * mcclellan_signal

        # --- Execute trades ---
        if combined > 0 and positions[ticker] == 0:
            # BUY
            cost = POSITION_SIZE * today_close
            if cash >= cost:
                positions[ticker] = POSITION_SIZE
                cash -= cost
                trade_log.append({
                    "date": date, "ticker": ticker, "action": "BUY",
                    "shares": POSITION_SIZE, "price": today_close, "cost": cost,
                })
        elif combined < 0 and positions[ticker] > 0:
            # SELL
            proceeds = positions[ticker] * today_close
            cash += proceeds
            trade_log.append({
                "date": date, "ticker": ticker, "action": "SELL",
                "shares": positions[ticker], "price": today_close, "proceeds": proceeds,
            })
            positions[ticker] = 0

    # Track portfolio value
    port_val = cash
    for t in TICKERS:
        if positions[t] > 0:
            port_val += positions[t] * prod_data[t].loc[date, "close"]
    portfolio_history.append({"date": date, "portfolio_value": port_val, "cash": cash})

    # Progress logging every 250 days
    if (day_idx - WARMUP_DAYS) % 250 == 0 and day_idx >= WARMUP_DAYS:
        print(f"Day {day_idx}/{len(all_dates)} | Portfolio: ${port_val:,.2f} | Trades: {len(trade_log)}")

print(f"\nBacktest complete! Total trades: {len(trade_log)}")

## Step 5: Results

In [ ]:
perf = pd.DataFrame(portfolio_history)
perf["date"] = pd.to_datetime(perf["date"])
perf = perf.set_index("date")

# Summary statistics
final_value = perf["portfolio_value"].iloc[-1]
total_return = (final_value / CAPITAL) - 1
days = (perf.index[-1] - perf.index[0]).days
annual_return = (1 + total_return) ** (365.25 / max(days, 1)) - 1

daily_returns = perf["portfolio_value"].pct_change().dropna()
sharpe = (daily_returns.mean() / daily_returns.std() * (252 ** 0.5)
          if daily_returns.std() > 0 else 0.0)

cummax = perf["portfolio_value"].cummax()
drawdown = (perf["portfolio_value"] - cummax) / cummax
max_drawdown = drawdown.min()

print("=" * 50)
print("BACKTEST RESULTS")
print("=" * 50)
print(f"Period:            {perf.index[0].date()} to {perf.index[-1].date()}")
print(f"Trading Days:      {len(perf)}")
print(f"Initial Capital:   ${CAPITAL:,.2f}")
print(f"Final Value:       ${final_value:,.2f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Annualized Return: {annual_return*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.4f}")
print(f"Max Drawdown:      {max_drawdown*100:.2f}%")
print(f"Total Trades:      {len(trade_log)}")
print("=" * 50)

# Equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

axes[0].plot(perf.index, perf["portfolio_value"], color="#2563eb", linewidth=1.5)
axes[0].axhline(y=CAPITAL, color="#94a3b8", linestyle="--", alpha=0.6, label="Initial Capital")
axes[0].set_title("Equity Curve — Endpoint Backtest", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Portfolio Value ($)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(perf.index, drawdown.values * 100, 0, color="#ef4444", alpha=0.3)
axes[1].plot(perf.index, drawdown.values * 100, color="#dc2626", linewidth=1)
axes[1].set_title("Drawdown", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Drawdown (%)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Trade log
if trade_log:
    trades_df = pd.DataFrame(trade_log)
    print(f"\nLast 10 trades:")
    print(trades_df.tail(10).to_string(index=False))

## Step 6: CLEANUP — Delete Endpoints

**IMPORTANT:** Always run this cell when done to avoid ongoing charges.
Each `ml.m5.large` endpoint costs ~$0.115/hour.

In [ ]:
for ticker in TICKERS:
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()
    for model_name in MODEL_NAMES:
        ep_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"
        model_name_full = f"{ep_name}-model"
        config_name = f"{ep_name}-config"

        # Delete endpoint
        try:
            sm_client.delete_endpoint(EndpointName=ep_name)
            print(f"Deleted endpoint: {ep_name}")
        except Exception as e:
            print(f"  Skip endpoint {ep_name}: {e}")

        # Delete endpoint config
        try:
            sm_client.delete_endpoint_config(EndpointConfigName=config_name)
            print(f"Deleted config: {config_name}")
        except Exception as e:
            print(f"  Skip config {config_name}: {e}")

        # Delete model
        try:
            sm_client.delete_model(ModelName=model_name_full)
            print(f"Deleted model: {model_name_full}")
        except Exception as e:
            print(f"  Skip model {model_name_full}: {e}")

print("\nAll endpoints cleaned up!")